In [1]:
import re
import string
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import nltk


In [2]:

nltk.download('punkt_tab')    # for tokenization
nltk.download('stopwords')    # for stopwords removal


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

## **Clean and normalize text for ML.**
Makes text more uniform, reduces noise, and simplifies vocabulary for model training.

In [4]:
def preprocess(text):
    text = text.lower()  # Lowercase
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\d+', '', text)  # Remove numbers
    tokens = text.split()

    negation_words = {"not", "no", "nor", "n't"}
    tokens = [word for word in tokens if word not in stop_words or word in negation_words]

    # tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords

    tokens = [stemmer.stem(word) for word in tokens]  # Stemming
    return ' '.join(tokens)

In [5]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [6]:


# csv_path = '/content/drive/My Drive/NLP/Restaurant_Reviews.tsv'


# .tsv to csv
# with open(csv_path, 'r', newline='') as tsvfile, open('/content/drive/My Drive/NLP/Restaurant_Reviews.csv', 'w', newline='') as csvfile:
#     tsv_reader = csv.reader(tsvfile, delimiter='\t')
#     csv_writer = csv.writer(csvfile, delimiter=',')
#     for row in tsv_reader:
#         csv_writer.writerow(row)


In [7]:
import csv

# df = pd.DataFrame(data)
df = pd.read_csv("/content/drive/My Drive/NLP/moviereviews.tsv", sep="\t")
df

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...
...,...,...
1995,pos,"i like movies with albert brooks , and i reall..."
1996,pos,it might surprise some to know that joel and e...
1997,pos,the verdict : spine-chilling drama from horror...
1998,pos,i want to correct what i wrote in a former ret...


In [8]:
missing_count = df['review'].isna().sum()
print(f"Missing reviews: {missing_count}")

print(df[df['review'].isna()])

Missing reviews: 35
     label review
140    pos    NaN
208    pos    NaN
270    neg    NaN
334    neg    NaN
448    neg    NaN
522    neg    NaN
606    pos    NaN
696    neg    NaN
728    pos    NaN
738    neg    NaN
744    pos    NaN
786    pos    NaN
820    pos    NaN
866    neg    NaN
986    neg    NaN
1072   neg    NaN
1156   pos    NaN
1204   pos    NaN
1260   neg    NaN
1272   neg    NaN
1294   neg    NaN
1314   pos    NaN
1330   neg    NaN
1332   neg    NaN
1426   pos    NaN
1456   pos    NaN
1486   pos    NaN
1538   neg    NaN
1568   neg    NaN
1584   pos    NaN
1588   pos    NaN
1600   pos    NaN
1662   pos    NaN
1904   pos    NaN
1908   neg    NaN


In [9]:
df = df.dropna(subset=['review'])
df['clean_review'] = df['review'].apply(preprocess)
df

/tmp/ipython-input-1312401290.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['clean_review'] = df['review'].apply(preprocess)


,label,review,clean_review
0,neg,how do films like mouse hunt get into theatres...,film like mous hunt get theatr isnt law someth...
1,neg,some talented actresses are blessed with a dem...,talent actress bless demonstr wide act rang ot...
2,pos,this has been an extraordinary year for austra...,extraordinari year australian film shine scoop...
3,pos,according to hollywood movies made in last few...,accord hollywood movi made last decad life sma...
4,neg,my first press screening of 1998 and already i...,first press screen alreadi ive gotten prime ca...
...,...,...,...
1995,pos,"i like movies with albert brooks , and i reall...",like movi albert brook realli like movi direct...
1996,pos,it might surprise some to know that joel and e...,might surpris know joel ethan coen brought una...
1997,pos,the verdict : spine-chilling drama from horror...,verdict spinechil drama horror maestro stephen...
1998,pos,i want to correct what i wrote in a former ret...,want correct wrote former retrospect david lea...


In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['label'])  # 'neg' becomes 0, 'pos' becomes 1


# **TF-IDF vectorisation**
converts text into numerical vectors

In [11]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_review'])
# y = df['label']

In [12]:
# y.unique()

In [13]:
y

array([0, 0, 1, ..., 1, 1, 1])

In [14]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 490732 stored elements and shape (1965, 30509)>

# **Model Training**

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [16]:
def train_and_evaluate_model(model, X_train, y_train, X_test, y_test, name="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n====== {name} ======")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    return model



In [17]:
lr_model = train_and_evaluate_model(LogisticRegression(), X_train, y_train, X_test, y_test, name="Logistic Regression")
nb_model = train_and_evaluate_model(MultinomialNB(), X_train, y_train, X_test, y_test, name="Naive Bayes")
svc_model = train_and_evaluate_model(SVC(kernel='linear'), X_train, y_train, X_test, y_test, name="Support Vector Classifier")



====== Logistic Regression ======
Accuracy: 0.8186440677966101
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.81      0.82       296
           1       0.81      0.83      0.82       294

    accuracy                           0.82       590
   macro avg       0.82      0.82      0.82       590
weighted avg       0.82      0.82      0.82       590


====== Naive Bayes ======
Accuracy: 0.8
Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.85      0.81       296
           1       0.83      0.75      0.79       294

    accuracy                           0.80       590
   macro avg       0.80      0.80      0.80       590
weighted avg       0.80      0.80      0.80       590


====== Support Vector Classifier ======
Accuracy: 0.823728813559322
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.81      0.82  

In [18]:
def predict_movie_review(text, model):
    clean = preprocess(text)
    vec = vectorizer.transform([clean])
    result = model.predict(vec)[0]
    return "Positive" if result == 1 else "Negative"


In [19]:
example_text_1 = "The movie was a waste of time"
example_text_2 = "This is a perfect movie."

for model, name in zip([lr_model, nb_model, svc_model], ["LogisticRegression", "NaiveBayes", "SVC"]):
    print(f"\n{name} Prediction 1: {predict_movie_review(example_text_1, model)}")
    print(f"{name} Prediction 2: {predict_movie_review(example_text_2, model)}")



LogisticRegression Prediction 1: Negative
LogisticRegression Prediction 2: Positive

NaiveBayes Prediction 1: Negative
NaiveBayes Prediction 2: Positive

SVC Prediction 1: Negative
SVC Prediction 2: Positive
